# Collin County TX — Full Geostack Pipeline

Demonstrates the complete geostack workflow for a CRE submarket analysis:

1. Setup & configuration
2. Data loading commands
3. ACS tract demographics (PostGIS)
4. LODES employment by tract (PostGIS)
5. Merge demographics + employment → jobs-per-resident
6. Employment choropleth map
7. Commute shed analysis
8. DuckDB top employment centers
9. Summary statistics table

## Cell 1 — Setup

In [ ]:
import os
import warnings

import geopandas as gpd
import pandas as pd
from rich.console import Console
from rich.table import Table

from geostack.census import get_county_acs_summary, get_tract_demographics
from geostack.db import get_engine
from geostack.duckdb_analytics import employment_by_sector, top_employment_centers
from geostack.lodes import commute_shed, tract_employment
from geostack.viz import choropleth_map, employment_map

warnings.filterwarnings("ignore")

STATE = "48"       # Texas FIPS
COUNTY = "085"     # Collin County FIPS
YEAR_ACS = 2023
YEAR_LODES = 2021  # latest LODES8 vintage

# Verify DATABASE_URL is configured
db_url = os.environ.get("DATABASE_URL")
if not db_url:
    raise EnvironmentError(
        "DATABASE_URL not set. Copy .env.example to .env and configure your PostGIS connection."
    )

engine = get_engine()
console = Console()
print(f"Connected to: {db_url.split('@')[-1]}")

## Cell 2 — Load Data

Run these commands once to populate PostGIS and the local parquet cache.
The cells below assume this has already been done.

```bash
# 1. Download TIGER/Line tract boundaries and load into PostGIS
geostack load-county --state 48 --county 085

# 2. Fetch ACS 5-year estimates and load into acs_tract_data
geostack load-acs --state 48 --county 085 --year 2023

# 3. Download LODES8 WAC + RAC + OD and load into PostGIS + parquet cache
geostack load-lodes --state 48 --county 085 --year 2021 --od
```

Or from a notebook cell using shell magic:
```python
!geostack load-county --state 48 --county 085
!geostack load-acs --state 48 --county 085 --year 2023
!geostack load-lodes --state 48 --county 085 --year 2021 --od
```

## Cell 3 — ACS Tract Demographics

In [ ]:
demo_gdf = get_tract_demographics(STATE, COUNTY, year=YEAR_ACS, engine=engine)

print(f"Tracts: {len(demo_gdf)}")
print(f"Columns: {list(demo_gdf.columns)}")
demo_gdf[["total_population", "median_household_income", "median_rent", "renter_occupied"]].head()

## Cell 4 — LODES Employment by Tract

In [ ]:
emp_df = tract_employment(STATE, COUNTY, YEAR_LODES, engine=engine)

print(f"Tracts: {len(emp_df)}  |  Total jobs: {emp_df['total_jobs'].sum():,.0f}")
emp_df[["tract_geoid", "total_jobs", "jobs_retail", "jobs_healthcare"]].head()

## Cell 5 — Merge: Jobs-per-Resident Ratio

In [ ]:
# Join on geoid (demo_gdf index) == tract_geoid (emp_df column)
merged = demo_gdf.join(emp_df.set_index("tract_geoid"), how="left")

# Compute jobs-per-resident
merged["jobs_per_resident"] = (
    merged["total_jobs"].fillna(0) / merged["total_population"].replace(0, float("nan"))
)

print(
    f"County-wide jobs/resident: "
    f"{merged['total_jobs'].sum() / merged['total_population'].sum():.2f}"
)
merged[["total_population", "total_jobs", "jobs_per_resident"]].describe().round(2)

## Cell 6 — Employment Choropleth Map

In [ ]:
# employment_map() handles the join and choropleth internally;
# we pass the already-merged GDF here for convenience
m = choropleth_map(
    merged.reset_index(),
    column="total_jobs",
    tooltip_cols=["geoid", "total_jobs", "total_population", "jobs_per_resident"],
    title="Total Jobs by Census Tract — Collin County TX",
    scheme="NaturalBreaks",
    k=6,
    cmap="YlOrRd",
)
m

## Cell 7 — Commute Shed: Top 10 Home Counties

In [ ]:
shed = commute_shed(STATE, COUNTY, YEAR_LODES, engine=engine)

top10 = shed.head(10)
print(f"Total home counties: {len(shed)}")
top10[["h_state_fips", "h_county_fips", "total_jobs"]].rename(
    columns={"total_jobs": "workers"}
)

## Cell 8 — DuckDB: Top Employment Centers (Block Level)

In [ ]:
top_blocks = top_employment_centers(STATE, YEAR_LODES, f"{STATE}{COUNTY}", n=10)

print("Top 10 census blocks by total jobs (DuckDB parquet path):")
top_blocks[["w_geocode", "tract_geoid", "total_jobs", "jobs_retail", "jobs_healthcare"]]


## Cell 9 — Summary Statistics Table

In [ ]:
acs_summary = get_county_acs_summary(STATE, COUNTY, year=YEAR_ACS, engine=engine)

table = Table(title=f"Collin County TX — ACS {YEAR_ACS} Summary", show_header=True)
table.add_column("Metric", style="bold")
table.add_column("Value", justify="right")

def fmt(val, is_currency=False, is_int=True):
    if pd.isna(val):
        return "—"
    if is_currency:
        return f"${val:,.0f}"
    if is_int:
        return f"{val:,.0f}"
    return f"{val:.1f}"

row = acs_summary.iloc[0]
table.add_row("Total Population",         fmt(row["total_population"]))
table.add_row("Total Housing Units",      fmt(row["total_housing_units"]))
table.add_row("Owner Occupied Units",     fmt(row["owner_occupied"]))
table.add_row("Renter Occupied Units",    fmt(row["renter_occupied"]))
table.add_row("Vacant Units",             fmt(row["vacant_units"]))
table.add_row("Median Age",               fmt(row["median_age"], is_int=False))
table.add_row("Median Household Income",  fmt(row["median_household_income"], is_currency=True))
table.add_row("Per Capita Income",        fmt(row["per_capita_income"], is_currency=True))
table.add_row("Median Rent",              fmt(row["median_rent"], is_currency=True))
table.add_row("Median Home Value",        fmt(row["median_home_value"], is_currency=True))
table.add_row("20+ Unit Buildings",       fmt(row["units_in_structure_20plus"]))

total_lodes_jobs = emp_df["total_jobs"].sum()
table.add_row(f"Total Jobs ({YEAR_LODES} LODES)", fmt(total_lodes_jobs))

console.print(table)